# 경구약제 이미지 인식 기능을 이용한 유저 헬스케어 데이터 수집 서비스_EDA



#### 1. 데이터 폴더별 항목 확인

`train_images`, `train_annotations`, `test_images`에서 항목 이름을 각각 최대 3개씩 출력

확인한 이미지 이름의 확장자는 `.png`였으며, 라벨 영역에는 `_json`으로 끝나는 항목이 존재

In [2]:
from pathlib import Path
from itertools import islice

data_dir = Path("../data/raw")

for folder_name in ["train_images", "train_annotations", "test_images"]:
    folder = data_dir / folder_name
    print(f"\n[{folder_name}]")
    for item in islice(folder.iterdir(), 3):
        print(item.name)


[train_images]
K-003483-025367-027733-035206_0_2_0_2_75_000_200.png
K-003483-025469-030308-035206_0_2_0_2_90_000_200.png
K-003351-013900-022074_0_2_0_2_75_000_200.png

[train_annotations]
K-003351-016232-033880_json
K-003351-035206-041768_json
K-003351-013900-029667_json

[test_images]
1186.png
412.png
374.png


#### 2. 라벨 폴더의 하위 구조 확인

> `train_annotations`의 `_json` 항목은 폴더인지 파일인지 확인


##### 결과

`_json`으로 끝나는 항목이 파일이 아닌 폴더임을 확인하고, 내부 항목을 조회

폴더 안에 하위 폴더가 한 번 더 있는 구조

```
train_annotations/
└── K-003351-016232-033880_json/
    ├── K-003351/
    ├── K-016232/
    └── K-033880/
```


In [3]:
from pathlib import Path
from itertools import islice

annotation_dir = Path("../data/raw/train_annotations")

for item in islice(annotation_dir.iterdir(), 3):
    print(f"\n{item.name}")
    print("종류:", "폴더" if item.is_dir() else "파일")

    if item.is_dir():
        for child in islice(item.iterdir(), 3):
            print("  └", child.name)


K-003351-016232-033880_json
종류: 폴더
  └ K-003351
  └ K-016232
  └ K-033880

K-003351-035206-041768_json
종류: 폴더
  └ K-003351
  └ K-041768
  └ K-035206

K-003351-013900-029667_json
종류: 폴더
  └ K-003351
  └ K-029667
  └ K-013900


#### 3. 샘플 JSON 파일의 최상위 구조 확인

라벨 폴더의 하위 경로에서 JSON 파일 하나를 찾아 읽고, 최상위 항목의 이름과 자료형을 확인

```
JSON 전체: dict
├── images        → 목록 (list)
├── type          → 문자열 (str)
├── annotations   → 목록 (list)
└── categories    → 목록 (list)
```

In [4]:
import json
from pathlib import Path

annotation_dir = Path("../data/raw/train_annotations")
json_path = next(annotation_dir.rglob("*.json"), None)

if json_path is None:
    print("JSON 파일을 찾지 못했어요.")
else:
    print("확인할 파일:", json_path.relative_to(annotation_dir))

    with json_path.open(encoding="utf-8-sig") as f:
        label = json.load(f)

    print("최상위 자료형:", type(label).__name__)

    if isinstance(label, dict):
        for key, value in label.items():
            print(f"- {key}: {type(value).__name__}")
    elif isinstance(label, list):
        print("항목 수:", len(label))
        if label and isinstance(label[0], dict):
            print("첫 항목의 키:", list(label[0].keys()))

확인할 파일: K-003351-016232-033880_json/K-003351/K-003351-016232-033880_0_2_0_2_75_000_200.json
최상위 자료형: dict
- images: list
- type: str
- annotations: list
- categories: list


#### 4. 샘플 JSON의 항목 개수 및 유형 확인

선택한 JSON 파일에는 이미지 정보 1개, 라벨 1개, 클래스 정보 1개가 포함

type의 실제 값은 `instances`

In [5]:
print("이미지 정보 개수:", len(label["images"]))
print("라벨 개수:", len(label["annotations"]))
print("클래스 정보 개수:", len(label["categories"]))
print("type의 실제 값:", label["type"])

이미지 정보 개수: 1
라벨 개수: 1
클래스 정보 개수: 1
type의 실제 값: instances
